# European Genome-phenome Archive (EGA) — Data Ingestion

**EGA** (European Genome-phenome Archive) is a permanent archive for personally identifiable genetic and phenotypic data from biomedical research. Operated jointly by EMBL-EBI and CRG, it is the primary repository for controlled-access human genomics data in Europe.

**Controlled-access model:** Raw sequence data, variant calls, and phenotype tables in EGA require approval from a Data Access Committee (DAC) before download. However, *study and dataset metadata* — titles, study types, technology platforms, and dataset counts — are freely queryable via the public REST API used here.

Key data types:
| Entity | Description |
|---|---|
| **Studies** | Top-level projects, each with an `EGAS` accession (e.g. `EGAS00001000037`) |
| **Datasets** | Subsets of a study released for access; each has a `EGAD` accession |
| **Samples** | Individual biological samples (`EGAN` accessions) |
| **Experiments** | Sequencing or array experiments (`EGAX` accessions) |
| **Runs** | Single instrument runs producing raw data files (`EGAR` accessions) |
| **Files** | Encrypted data files (`EGAF` accessions); only accessible after DAC approval |

**API base:** `https://ega-archive.org/metadata/v2/`

**Reference:** Freeberg et al. (2022), *Nucleic Acids Research*, The European Genome-phenome Archive in 2021

# TODO

* [x] **Ingest data**
    * [x] Connect to EGA metadata REST API and fetch a single study record
    * [x] Page through public study metadata and cache to `data/ega_studies.json`
    * [x] Parse studies into a Polars DataFrame (study_id, title, study_type, submission_date, dataset_count, technology)
    * [x] Fetch datasets for a study of interest; parse into a datasets DataFrame
    * [x] Print shape, dtypes, and head for both DataFrames
* [ ] **Explore and clean**
    * [ ] Summarize study counts by study_type and technology platform
    * [ ] Examine submission date trends (studies deposited per year)
    * [ ] Inspect dataset access statuses and file count distributions
* [ ] **Analysis**
    * [ ] Identify the most common disease areas and technology platforms
    * [ ] Analyse the relationship between dataset count and study size
    * [ ] Cross-reference EGA studies with publications (PubMed IDs)
* [ ] **Visualization**
    * [ ] Bar chart of study counts by type and technology
    * [ ] Timeline of EGA submissions over the years
    * [ ] Dataset file count distribution (histogram)
* [ ] **Statistical analysis**
    * [ ] Discuss the controlled-access model and its implications for data completeness
    * [ ] Analyse metadata completeness rates across fields
    * [ ] Discuss power considerations when working with DAC-approved subsets

In [ ]:
import requests
import time
import json
from pathlib import Path

import polars as pl

## 1. Ingest Data

### 1.1 Connect to EGA API and Fetch a Single Study

In [ ]:
EGA_BASE = "https://ega-archive.org/metadata/v2"
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

def ega_get(endpoint: str, params: dict = None) -> dict:
    """
    Send a GET request to the EGA metadata REST API.

    A 0.5-second delay is applied after every call as a courtesy to
    the EGA servers (no formal rate-limit is published, but polite
    crawling is required by the terms of use).

    Parameters
    ----------
    endpoint : str
        API path relative to EGA_BASE (e.g. "studies/EGAS00001000037").
    params : dict, optional
        Query parameters (e.g. ``{"limit": 100, "offset": 0}``).

    Returns
    -------
    dict
        Parsed JSON response body.

    Raises
    ------
    requests.HTTPError
        If the server returns a 4xx or 5xx status code.
    """
    url = f"{EGA_BASE}/{endpoint}"
    resp = requests.get(url, params=params or {}, timeout=30)
    resp.raise_for_status()
    time.sleep(0.5)   # polite delay between requests
    return resp.json()

# Connectivity check: fetch a single well-known study
FOCUS_STUDY = "EGAS00001000037"
study = ega_get(f"studies/{FOCUS_STUDY}")

print(f"Study ID    : {study.get('egaStableId') or study.get('id')}")
print(f"Title       : {study.get('title')}")
print(f"Study type  : {study.get('studyType')}")
print(f"Description : {(study.get('description') or '')[:200]}")

# Publications linked to this study
pubs = study.get("publications") or []
for pub in pubs:
    print(f"  Publication: {pub.get('pubmedId') or pub.get('doi') or pub}")

### 1.2 Page Through Public Study Metadata and Cache to Disk

In [ ]:
STUDIES_CACHE = DATA_DIR / "ega_studies.json"
PAGE_SIZE = 100   # maximum the EGA API accepts per page

def fetch_all_studies(cache_path: Path = STUDIES_CACHE) -> list[dict]:
    """
    Page through the EGA /studies endpoint and return all study records.

    Results are written to ``cache_path`` on first run; subsequent calls
    return the cached data immediately, avoiding redundant API traffic.

    Parameters
    ----------
    cache_path : Path
        File path for the JSON cache.

    Returns
    -------
    list[dict]
        One dict per study containing all metadata fields returned by the API.
    """
    if cache_path.exists():
        print(f"Loading from cache: {cache_path}")
        return json.loads(cache_path.read_text())

    all_studies: list[dict] = []
    offset = 0

    while True:
        resp = ega_get("studies", {"limit": PAGE_SIZE, "offset": offset})

        # The API returns a list at the top level or nested under a key;
        # handle both shapes defensively.
        if isinstance(resp, list):
            batch = resp
        else:
            batch = resp.get("response", {}).get("result", resp.get("result", []))

        if not batch:
            break

        all_studies.extend(batch)
        print(f"  Fetched {len(all_studies):>5} studies (offset {offset})", end="\r")
        offset += PAGE_SIZE

        # Stop when we receive a partial page — we've reached the end
        if len(batch) < PAGE_SIZE:
            break

    print(f"\nDone. Total studies fetched: {len(all_studies)}")
    cache_path.write_text(json.dumps(all_studies, indent=2))
    print(f"Cached to {cache_path}")
    return all_studies

studies_raw = fetch_all_studies()
print(f"Total records: {len(studies_raw)}")

### 1.3 Parse Studies into a Polars DataFrame

In [ ]:
def flatten_study(s: dict) -> dict:
    """
    Flatten a single raw EGA study record into a row-friendly dict.

    The EGA API nests some fields (e.g. publications as a list of dicts,
    technology inside an ``attributes`` block). This function extracts the
    key scalar fields and encodes list fields as pipe-separated strings.

    Parameters
    ----------
    s : dict
        Raw study record from the EGA metadata API.

    Returns
    -------
    dict
        Flat dict suitable for building a Polars DataFrame row.
    """
    # Stable accession may live under different keys across API versions
    study_id = (
        s.get("egaStableId")
        or s.get("stableId")
        or s.get("id")
        or s.get("accessionId")
    )

    # Submission/creation date: try several field names
    sub_date = (
        s.get("submissionDate")
        or s.get("creationTime")
        or s.get("firstPublished")
    )

    # Technology platform: may be a top-level field or nested in attributes
    technology = s.get("technology") or s.get("studyTechnology")
    if technology is None:
        attrs = s.get("attributes") or []
        for attr in attrs:
            if isinstance(attr, dict) and str(attr.get("tag", "")).lower() == "technology":
                technology = attr.get("value")
                break

    # Dataset count: number of EGAD datasets linked to this study
    datasets = s.get("datasets") or s.get("datasetsCount") or []
    dataset_count = len(datasets) if isinstance(datasets, list) else int(datasets or 0)

    # Publications: collect PubMed IDs as a pipe-separated string
    pubs = s.get("publications") or []
    pubmed_ids = " | ".join(
        str(p.get("pubmedId") or p.get("doi") or "")
        for p in pubs
        if isinstance(p, dict)
    ).strip(" |") or None

    return {
        "study_id":        study_id,
        "title":           s.get("title"),
        "study_type":      s.get("studyType") or s.get("type"),
        "submission_date": str(sub_date)[:10] if sub_date else None,  # keep YYYY-MM-DD only
        "dataset_count":   dataset_count,
        "technology":      technology,
        "pubmed_ids":      pubmed_ids,
    }

rows = [flatten_study(s) for s in studies_raw]

studies = pl.DataFrame(rows).with_columns(
    # Parse the ISO date string; use strict=False so nulls stay null rather than erroring
    pl.col("submission_date").str.to_date("%Y-%m-%d", strict=False),
    pl.col("dataset_count").cast(pl.Int32),
)

print(f"Shape  : {studies.shape}")
print(f"Memory : {studies.estimated_size('kb'):.0f} KB")
print("\nDtypes:")
print(studies.dtypes)
print()
studies.head(5)

### 1.4 Fetch Datasets for a Study of Interest

In [ ]:
def fetch_study_datasets(study_accession: str) -> pl.DataFrame:
    """
    Fetch all datasets belonging to a single EGA study.

    Datasets (``EGAD`` accessions) are the units of access control in EGA —
    each dataset has its own DAC and access status. Metadata (accession, data
    type, file count, access status) is public even without DAC approval.

    Parameters
    ----------
    study_accession : str
        EGA study accession, e.g. ``"EGAS00001000037"``.

    Returns
    -------
    pl.DataFrame
        One row per dataset with columns: dataset_id, study_id, data_type,
        file_count, access_status.
    """
    resp = ega_get(f"studies/{study_accession}/datasets")

    # API may return a bare list or a wrapped object
    if isinstance(resp, list):
        raw_datasets = resp
    else:
        raw_datasets = resp.get("response", {}).get("result", resp.get("result", []))

    rows = []
    for ds in raw_datasets:
        rows.append({
            "dataset_id":     ds.get("egaStableId") or ds.get("stableId") or ds.get("id"),
            "study_id":       study_accession,
            "data_type":      ds.get("datasetType") or ds.get("type"),
            "file_count":     ds.get("numberOfFiles") or ds.get("fileCount") or 0,
            "access_status":  ds.get("status") or ds.get("accessType") or "controlled",
        })

    return pl.DataFrame(rows).with_columns(
        pl.col("file_count").cast(pl.Int32, strict=False),
    )

datasets = fetch_study_datasets(FOCUS_STUDY)

print(f"Study  : {FOCUS_STUDY}")
print(f"Shape  : {datasets.shape}")
print()
datasets.head(10)

### 1.5 Summary — Shape, Dtypes, and Head for Both DataFrames

In [ ]:
for name, df in [("studies", studies), ("datasets", datasets)]:
    print(f"{'='*50}")
    print(f"DataFrame : {name}")
    print(f"Shape     : {df.shape[0]:,} rows × {df.shape[1]} columns")
    print(f"Memory    : {df.estimated_size('kb'):.1f} KB")
    print(f"\nDtypes:")
    for col, dtype in zip(df.columns, df.dtypes):
        null_pct = df[col].null_count() / len(df) * 100
        print(f"  {col:<20} {str(dtype):<15} nulls: {null_pct:.1f}%")
    print(f"\nHead:")
    print(df.head(3))
    print()